# Betting Data Analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data.csv')
print(f'Dataset: {df.shape[0]} bets, {df.shape[1]} columns')
df.head()

Dataset: 1117 bets, 77 columns


,Unique Bet Rule 1,Team,Total?,Unnamed: 3,Bet,Book,Date,Market,Other,LineTaken,...,CZR M,CZR O,BM M$,BM O$,DK M,DK O,4C M,4C M$,4C O,4C O$
0,1,Pistons,0,2024-10,Pistons +30.5 L,FWmb,2024-10-26,-1400,630,-500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Pistons,0,2024-10,Pistons +32.5 L,FWmb,2024-10-26,-1450,640,-480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Pistons,0,2024-10,Pistons +29.5 L,FWmb,2024-10-26,-750,430,-278,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,Clippers,0,2024-10,Clippers ml L,FWmb,2024-10-31,-199,169,-165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,Clippers,0,2024-10,Clippers ml L,BRmb,2024-10-31,-259,198,-200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# --- Data Cleanup ---

# 1. Strip whitespace from column names
df.columns = df.columns.str.strip()

# 2. Convert Date columns to datetime
df['Date'] = pd.to_datetime(df['Date'])
df['Date.1'] = pd.to_datetime(df['Date.1'])

# 3. Drop Time column
df.drop(columns=['Time'], inplace=True)

# 4. Rename Unnamed: 3 to Month
df.rename(columns={'Unnamed: 3': 'Month'}, inplace=True)

# 5. Rename Unnamed: 16 to BetType
df.rename(columns={'Unnamed: 16': 'BetType'}, inplace=True)

# 6. Total? to bool
df['Total?'] = df['Total?'].astype(bool)

# 7. Grade to categorical
df['Grade'] = df['Grade'].astype('category')

# 8. Drop Ignore columns
df.drop(columns=['Ignore', 'Ignore.1'], inplace=True)

# 9. Drop 100% null columns
null_cols = [c for c in df.columns if df[c].isna().all()]
print(f'Dropping {len(null_cols)} fully null columns: {null_cols}')
df.drop(columns=null_cols, inplace=True)

print(f'\nCleaned: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nDtypes:\n{df.dtypes.to_string()}')

Dropping 8 fully null columns: ['CloseLine', 'CloseOther', 'CloseEdge', 'Trueline', '4C M', '4C M$', '4C O', '4C O$']

Cleaned: 1117 rows, 66 columns

Dtypes:
Unique Bet Rule 1             int64
Team                         object
Total?                         bool
Month                        object
Bet                          object
Book                         object
Date                 datetime64[ns]
Market                        int64
Other                         int64
LineTaken                     int64
Edge                        float64
TrueL                       float64
Decim                       float64
Betsize                     float64
Sport                        object
BetType                      object
Opp                          object
Total Stake                 float64
Rich Stake                  float64
Other Stake                 float64
Grade                      category
Net                         float64
ExpWinPlace                 float64
Date.1       

In [7]:
# --- Analysis DataFrame ---

# Filter to Basketball only
bdf = df[df['Sport'] == 'Basketball'].copy()
print(f'Filtered to Basketball: {len(bdf)} rows')

# Select and rename columns
bdf = bdf[['Unique Bet Rule 1', 'Team', 'Total?', 'Bet', 'Date', 'Market',
            'Other', 'LineTaken', 'Edge', 'BetType', 'Rich Stake', 'Grade']].copy()

bdf.rename(columns={
    'Unique Bet Rule 1': 'Bet',
    'Bet': 'ExactBet',
    'Total?': 'Total',
    'Rich Stake': 'RichStake',
    'LineTaken': 'LineTaken',
}, inplace=True)

# Convert Edge from decimal to percentage with 1 decimal place
bdf['Edge'] = (bdf['Edge'] * 100).round(1)

# Round RichStake to 2 decimal places to fix floating point precision
bdf['RichStake'] = bdf['RichStake'].round(2)

# Calculate Expected Profit
bdf['ExpProfit'] = (bdf['RichStake'] * bdf['Edge'] / 100).round(0).astype(int)

# Calculate Profit
def calc_profit(row):
    if row['Grade'] == 'P':
        return 0.0
    if row['Grade'] == 'L':
        return -row['RichStake']
    # Grade == W: calculate winnings from american odds
    odds = row['LineTaken']
    stake = row['RichStake']
    if odds > 0:
        return stake * (odds / 100)
    else:
        return stake * (100 / abs(odds))

bdf['Profit'] = bdf.apply(calc_profit, axis=1).astype(int)

bdf.to_csv('clean_data.csv', index=False)
print(f'{len(bdf)} bets | Total Profit: ${bdf["Profit"].sum():,}')
print('Exported to clean_data.csv')
bdf.head(10)

Filtered to Basketball: 571 rows
571 bets | Total Profit: $18,588
Exported to clean_data.csv


,Bet,Team,Total,ExactBet,Date,Market,Other,LineTaken,Edge,BetType,RichStake,Grade,ExpProfit,Profit
534,221,Purdue,False,Purdue ml L,2025-12-06,112,-142,133,3.8,moneyline,525.0,L,20,-525
545,227,Utah,False,Utah Jazz ml L,2025-12-08,1953,-11372,15000,607.2,moneyline,20.0,L,121,-20
548,230,Denver,True,Denver Nuggets u253.5 L,2025-12-11,103,128,120,16.4,total,130.0,W,21,156
549,231,North,True,North Dakota State u149.5 L,2025-12-11,-142,107,104,11.9,total,230.0,W,27,239
550,232,Butler,False,Butler ml L,2025-12-13,-103,-123,133,11.6,moneyline,882.0,W,102,1173
551,233,Cal,True,Cal Riverside u157.5 L,2025-12-13,-115,-102,125,15.7,total,112.0,W,18,140
552,234,Central,False,Central Arkansas +25.5 L,2025-12-13,-118,-112,130,16.4,spread,1769.0,W,290,2299
553,234,Central,False,Central Arkansas +15.5 L,2025-12-13,-115,-112,120,10.7,spread,684.0,W,73,820
554,235,Central,True,Central Arkansas u153.5 L,2025-12-13,-110,-122,128,11.3,total,384.0,L,43,-384
555,236,Chattanooga,True,Chattanooga o164.5 L,2025-12-13,-123,-108,116,11.3,total,606.0,W,68,702


In [8]:
# --- Merge rows by Bet ID ---

mdf = pd.read_csv('clean_data.csv')

def grade_agg(grades):
    unique = grades.unique()
    if len(unique) == 1:
        return unique[0]
    return 'mix'

merged = mdf.groupby('Bet').agg(
    Team=('Team', 'first'),
    Total=('Total', 'first'),
    ExactBet=('ExactBet', 'first'),
    Date=('Date', 'first'),
    Market=('Market', 'median'),
    Other=('Other', 'median'),
    LineTaken=('LineTaken', 'median'),
    Edge=('Edge', 'median'),
    BetType=('BetType', 'first'),
    RichStake=('RichStake', 'sum'),
    Grade=('Grade', grade_agg),
    Profit=('Profit', 'sum'),
).reset_index()

# Fix floating point precision
merged['Market'] = merged['Market'].astype(int)
merged['Other'] = merged['Other'].astype(int)
merged['LineTaken'] = merged['LineTaken'].astype(int)
merged['Edge'] = merged['Edge'].round(1)
merged['RichStake'] = merged['RichStake'].round(2)

merged.to_csv('merged_data.csv', index=False)
print(f'Merged: {len(mdf)} rows -> {len(merged)} bets')
print(f'Grade breakdown: {merged["Grade"].value_counts().to_dict()}')
print('Exported to merged_data.csv')
merged.head(10)

Merged: 571 rows -> 373 bets
Grade breakdown: {'L': 182, 'W': 164, 'mix': 26, 'P': 1}
Exported to merged_data.csv


,Bet,Team,Total,ExactBet,Date,Market,Other,LineTaken,Edge,BetType,RichStake,Grade,Profit
0,221,Purdue,False,Purdue ml L,2025-12-06,112,-142,133,3.8,moneyline,525.0,L,-525
1,227,Utah,False,Utah Jazz ml L,2025-12-08,1953,-11372,15000,607.2,moneyline,20.0,L,-20
2,230,Denver,True,Denver Nuggets u253.5 L,2025-12-11,103,128,120,16.4,total,130.0,W,156
3,231,North,True,North Dakota State u149.5 L,2025-12-11,-142,107,104,11.9,total,230.0,W,239
4,232,Butler,False,Butler ml L,2025-12-13,-103,-123,133,11.6,moneyline,882.0,W,1173
5,233,Cal,True,Cal Riverside u157.5 L,2025-12-13,-115,-102,125,15.7,total,112.0,W,140
6,234,Central,False,Central Arkansas +25.5 L,2025-12-13,-116,-112,125,13.6,spread,2453.0,W,3119
7,235,Central,True,Central Arkansas u153.5 L,2025-12-13,-110,-122,128,11.3,total,384.0,L,-384
8,236,Chattanooga,True,Chattanooga o164.5 L,2025-12-13,-123,-108,116,11.3,total,606.0,W,702
9,237,Dayton,False,Dayton -36.5 L,2025-12-13,-114,-110,116,8.9,spread,606.0,L,-606


In [12]:
# --- Performance Analysis by Bucket ---

cdf = pd.read_csv('clean_data.csv')

# Exclude pushes from analysis
cdf = cdf[cdf['Grade'] != 'P'].copy()

def summarize(group):
    """Return performance summary for a group of bets."""
    staked = group['RichStake'].sum()
    exp_profit = group['ExpProfit'].sum()
    return pd.Series({
        'Bets': int(len(group)),
        'Wins': int((group['Grade'] == 'W').sum()),
        'Win%': round((group['Grade'] == 'W').mean() * 100, 1),
        'Staked': int(staked),
        'ExpProfit': int(exp_profit),
        'Profit': int(group['Profit'].sum()),
        'ExpROI%': round(exp_profit / staked * 100, 1),
        'ROI%': round(group['Profit'].sum() / staked * 100, 1),
    })

def int_cols(summary):
    """Cast integer columns."""
    for col in ['Bets', 'Wins', 'Staked', 'ExpProfit', 'Profit']:
        summary[col] = summary[col].astype(int)
    return summary

# --- 1. By Odds Bands (LineTaken) ---
odds_bins = [-999999, -150, -100, 150, 999999]
odds_labels = ['-150 or less', '-150 to -100', '+100 to +150', '+150 or greater']
cdf['OddsBand'] = pd.cut(cdf['LineTaken'], bins=odds_bins, labels=odds_labels)

print('=== Performance by Odds Band ===')
odds_summary = cdf.groupby('OddsBand', observed=False).apply(summarize, include_groups=False).reset_index()
display(int_cols(odds_summary))

# --- 2. By Edge Bands ---
edge_bins = [0, 5, 10, 15, float('inf')]
edge_labels = ['0-5%', '5-10%', '10-15%', '15%+']
cdf['EdgeBand'] = pd.cut(cdf['Edge'], bins=edge_bins, labels=edge_labels)

print('\n=== Performance by Edge Band ===')
edge_summary = cdf.groupby('EdgeBand', observed=False).apply(summarize, include_groups=False).reset_index()
display(int_cols(edge_summary))

# --- 3. By BetType ---
print('\n=== Performance by BetType ===')
core_types = cdf[cdf['BetType'].isin(['moneyline', 'spread', 'total'])]
bettype_summary = core_types.groupby('BetType').apply(summarize, include_groups=False).reset_index()
display(int_cols(bettype_summary))

# --- 4. Grand Totals ---
print('\n=== Grand Totals ===')
totals = summarize(cdf).to_frame().T
display(int_cols(totals))

=== Performance by Odds Band ===


,OddsBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,-150 or less,14,9,64.3,30398,2367,1228,7.8,4.0
1,-150 to -100,94,56,59.6,112812,10910,-1896,9.7,-1.7
2,+100 to +150,363,188,51.8,247393,28187,21566,11.4,8.7
3,+150 or greater,99,31,31.3,42905,8442,-2310,19.7,-5.4



=== Performance by Edge Band ===


,EdgeBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,0-5%,23,13,56.5,27086,774,-5043,2.9,-18.6
1,5-10%,166,88,53.0,147068,12211,16183,8.3,11.0
2,10-15%,209,106,50.7,152996,18307,5058,12.0,3.3
3,15%+,141,62,44.0,84612,19394,6261,22.9,7.4



=== Performance by BetType ===


,BetType,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,moneyline,148,66,44.6,113002,14561,14294,12.9,12.6
1,spread,237,119,50.2,217302,23902,4580,11.0,2.1
2,total,179,96,53.6,99720,11119,-511,11.2,-0.5



=== Grand Totals ===


,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,570,284,49.8,433510,49906,18588,11.5,4.3


In [19]:
# --- Export Performance Tables to CSV ---

blank = pd.DataFrame([{}])

odds_export = odds_summary.rename(columns={'OddsBand': 'Bucket'}).copy()
edge_export = edge_summary.rename(columns={'EdgeBand': 'Bucket'}).copy()
bettype_export = bettype_summary.rename(columns={'BetType': 'Bucket'}).copy()
totals_export = totals.assign(Bucket='All').copy()
stake_export = stake_summary.rename(columns={'StakeBand': 'Bucket'}).copy()

odds_export.insert(0, 'Group', 'Odds Band')
edge_export.insert(0, 'Group', 'Edge Band')
bettype_export.insert(0, 'Group', 'Bet Type')
totals_export.insert(0, 'Group', 'Grand Total')
stake_export.insert(0, 'Group', 'Stake Band')

combined = pd.concat([
    odds_export,
    blank,
    edge_export,
    blank,
    bettype_export,
    blank,
    totals_export,
    blank,
    blank,
    stake_export,
], ignore_index=True)

# Reorder so Group and Bucket are first
cols = ['Group', 'Bucket'] + [c for c in combined.columns if c not in ('Group', 'Bucket')]
combined = combined[cols]

combined.to_csv('performance_summary.csv', index=False)
print(f'Exported to performance_summary.csv')
combined

Exported to performance_summary.csv


,Group,Bucket,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,Odds Band,-150 or less,14.0,9.0,64.3,30398.0,2367.0,1228.0,7.8,4.0
1,Odds Band,-150 to -100,94.0,56.0,59.6,112812.0,10910.0,-1896.0,9.7,-1.7
2,Odds Band,+100 to +150,363.0,188.0,51.8,247393.0,28187.0,21566.0,11.4,8.7
3,Odds Band,+150 or greater,99.0,31.0,31.3,42905.0,8442.0,-2310.0,19.7,-5.4
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Edge Band,0-5%,23.0,13.0,56.5,27086.0,774.0,-5043.0,2.9,-18.6
6,Edge Band,5-10%,166.0,88.0,53.0,147068.0,12211.0,16183.0,8.3,11.0
7,Edge Band,10-15%,209.0,106.0,50.7,152996.0,18307.0,5058.0,12.0,3.3
8,Edge Band,15%+,141.0,62.0,44.0,84612.0,19394.0,6261.0,22.9,7.4
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# --- Performance by Stake Band (Merged Data) ---

mdf = pd.read_csv('merged_data.csv')
mdf = mdf[mdf['Grade'] != 'P'].copy()

# Add ExpProfit (Edge is already a percentage)
mdf['ExpProfit'] = (mdf['RichStake'] * mdf['Edge'] / 100).round(0).astype(int)

stake_bins = [0, 500, 1000, 2000, float('inf')]
stake_labels = ['0-500', '500-1000', '1000-2000', '2000+']
mdf['StakeBand'] = pd.cut(mdf['RichStake'], bins=stake_bins, labels=stake_labels)

def summarize(group):
    staked = group['RichStake'].sum()
    exp_profit = group['ExpProfit'].sum()
    return pd.Series({
        'Bets': int(len(group)),
        'Wins': int((group['Grade'] == 'W').sum()),
        'Win%': round((group['Grade'] == 'W').mean() * 100, 1),
        'Staked': int(staked),
        'ExpProfit': int(exp_profit),
        'Profit': int(group['Profit'].sum()),
        'ExpROI%': round(exp_profit / staked * 100, 1),
        'ROI%': round(group['Profit'].sum() / staked * 100, 1),
    })

def int_cols(summary):
    for col in ['Bets', 'Wins', 'Staked', 'ExpProfit', 'Profit']:
        summary[col] = summary[col].astype(int)
    return summary

print('=== Performance by Stake Band (Merged) ===')
stake_summary = mdf.groupby('StakeBand', observed=False).apply(summarize, include_groups=False).reset_index()
display(int_cols(stake_summary))

=== Performance by Stake Band (Merged) ===


,StakeBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,0-500,155,70,45.2,42071,5637,4490,13.4,10.7
1,500-1000,107,52,48.6,73252,7904,12041,10.8,16.4
2,1000-2000,55,21,38.2,78409,10757,-3575,13.7,-4.6
3,2000+,55,21,38.2,239776,27048,5632,11.3,2.3
